# v16.4 — D-1 dan v3.1 di NSRDB dan Georgia 12-lead (data yang belum pernah dipakai)

Rancangan dikunci sebelum sinyal dibaca: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.4_praregistrasi.log`). Logika di
`uji_baru.py`; mesin prediksi dan evaluasi dari v16.1–v16.3.

- **NSRDB**: 18 orang sehat, Holter ±24 jam → alarm AF palsu sepanjang hari (H4).
- **Georgia**: 10.344 EKG 12-lead klinis (lead I) dengan label dokter → D-1 vs v3.1 (H1, H2), sinus takikardia (H3), AFL, sinus
  bradikardia.

In [1]:
import json, sys, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import uji_baru as U
sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
R = {}
for ds in U.DATASET:
    U.jalankan(ds)        # dilewati bila cache per rekaman sudah lengkap
Z = {}
for ds in U.DATASET:
    Z[ds], info = U.muat(ds)
    R.setdefault("data", {})[ds] = info
    print(ds, json.dumps(info, ensure_ascii=False))

  Georgia: 1034/10344 rekaman


  Georgia: 2068/10344 rekaman


  Georgia: 3102/10344 rekaman


  Georgia: 4136/10344 rekaman


  Georgia: 5170/10344 rekaman


  Georgia: 6204/10344 rekaman


... (2 outputs left out)

  Georgia: 9306/10344 rekaman


  Georgia: 10340/10344 rekaman


NSRDB {"rekaman": 18, "window": 314972, "window_lolos": 294162, "per_kelompok": {"non-AF": [18, 314972, 294162]}}


Georgia {"rekaman": 10344, "window": 10344, "window_lolos": 10249, "per_kelompok": {"sinus normal": [1752, 1752, 1734], "sinus bradikardia": [1656, 1656, 1648], "AF/AFL": [741, 741, 725], "lainnya": [4951, 4951, 4909], "sinus takikardia": [1244, 1244, 1233]}}


## 1. Kendali mutu: jumlah rekaman per kelompok sama dengan header

In [2]:
from collections import Counter
hdr = Counter(U.kelompok_georgia(U.dx_georgia(U.AKAR["Georgia"] / r)) for r in U.rekaman("Georgia"))
pk = R["data"]["Georgia"]["per_kelompok"]
assert {g: v[0] for g, v in pk.items()} == dict(hdr), (pk, hdr)
assert R["data"]["NSRDB"]["rekaman"] == 18
display(pd.DataFrame({g: {"rekaman": v[0], "window": v[1], "lolos cek": v[2], "% lolos": round(100 * v[2] / max(v[1], 1), 1)}
                      for g, v in pk.items()}).T)
jam = R["data"]["NSRDB"]["window_lolos"] * 5 / 3600
print(f"NSRDB: {R['data']['NSRDB']['window_lolos']} / {R['data']['NSRDB']['window']} window lolos cek (±{jam:.0f} jam)")

,rekaman,window,lolos cek,% lolos
sinus normal,1752.0,1752.0,1734.0,99.0
sinus bradikardia,1656.0,1656.0,1648.0,99.5
AF/AFL,741.0,741.0,725.0,97.8
lainnya,4951.0,4951.0,4909.0,99.2
sinus takikardia,1244.0,1244.0,1233.0,99.1


NSRDB: 294162 / 314972 window lolos cek (±409 jam)


## 2. NSRDB (H4): alarm AF palsu pada orang sehat sepanjang hari

In [3]:
N = U.evaluasi_nsrdb(Z["NSRDB"])
R["nsrdb"] = N
display(pd.DataFrame({k: {"alarm (% window)": f"{v['alarm']*100:.2f}% [{v['ci'][0]*100:.2f}–{v['ci'][1]*100:.2f}]",
                          "tanpa artefak": f"{N['alarm_tanpa_artefak'][k]['alarm']*100:.2f}%",
                          "episode palsu/jam": f"{N['episode'][k]['palsu_per_jam']:.2f}"} for k, v in N["alarm"].items()}).T)
print("Δ D-1 − v3.1:", N["banding_d1_v31"], "→ H4", "lolos" if N["kriteria"]["H4_nsrdb"] else "GAGAL")
display(pd.DataFrame(N["per_subjek"]).T.round(4))
display(pd.DataFrame({k: {b: (f"{v['alarm']*100:.1f}% (n={v['n']})" if v["alarm"] is not None else "–") for b, v in p.items()}
                      for k, p in N["pita"].items()}).T)

,alarm (% window),tanpa artefak,episode palsu/jam
v3.1 paket,15.00% [10.90–19.96],15.30%,51.89
v1 paket,22.17% [16.59–28.57],22.37%,70.71
D-0,2.08% [1.63–2.58],2.30%,10.63
D-1,5.64% [4.30–7.15],5.57%,20.55


Δ D-1 − v3.1: {'delta': -0.09364227874436536, 'ci': [-0.14590649645859652, -0.05315313755638057], 'p_kurang': 1.0} → H4 lolos


,v3.1 paket,D-1,D-0,v1 paket,n
16265,0.1804,0.0500,0.0226,0.2850,17087.0
16272,0.0494,0.0303,0.0119,0.0536,16698.0
16273,0.1841,0.0730,0.0275,0.2397,16337.0
16420,0.0766,0.0402,0.0168,0.1219,16404.0
16483,0.0989,0.0827,0.0343,0.1630,17031.0
16539,0.4539,0.0167,0.0068,0.6063,17278.0
16773,0.2080,0.1462,0.0188,0.2885,16470.0
16786,0.0461,0.0161,0.0069,0.0738,17198.0
16795,0.3233,0.0537,0.0210,0.4223,15906.0
17052,0.1030,0.0933,0.0137,0.1643,15879.0


,< 100,100–110,110–120,≥ 120
v3.1 paket,13.7% (n=275995),22.2% (n=13201),59.1% (n=3798),97.2% (n=1168)
v1 paket,20.6% (n=275995),28.9% (n=13201),91.1% (n=3798),97.3% (n=1168)
D-0,1.7% (n=275995),7.6% (n=13201),10.6% (n=3798),8.7% (n=1168)
D-1,4.8% (n=275995),17.1% (n=13201),25.0% (n=3798),22.3% (n=1168)


## 3. Georgia (H1–H3): EKG klinis 12-lead, lead I

In [4]:
G = U.evaluasi_georgia(Z["Georgia"])
R["georgia"] = G
display(pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.3f}–{v[m][2]:.3f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")}
                      for k, v in G["titik"].items()}).T)
display(pd.DataFrame({k: {m: f"{v[m][0]:+.3f} [{v[m][1]:+.3f}; {v[m][2]:+.3f}] P={v[m][3]:.3f}" for m in ("Macro-F1", "Se", "Sp", "PPV")}
                      for k, v in G["banding"].items()}).T)
display(pd.DataFrame({g: {k: f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.0f}–{v['ci'][1]*100:.0f}] (n={v['n']})" for k, v in d.items()}
                      for g, d in {**G["kelompok"], **G["subkelompok"]}.items()}).T)
display(pd.DataFrame({k: {b: (f"{v['alarm']*100:.1f}% (n={v['n']})" if v["alarm"] is not None else "–") for b, v in p.items()}
                      for k, p in G["pita"].items()}).T)
print(G["kriteria"])

,Macro-F1,Se,Sp,PPV
v3.1 paket,0.691 [0.677–0.704],0.770 [0.736–0.799],0.877 [0.870–0.884],0.323 [0.302–0.345]
D-1,0.843 [0.828–0.856],0.706 [0.673–0.739],0.978 [0.975–0.981],0.709 [0.676–0.742]
D-0,0.860 [0.845–0.873],0.646 [0.609–0.678],0.992 [0.990–0.994],0.857 [0.828–0.885]
v1 paket,0.664 [0.652–0.677],0.786 [0.755–0.817],0.850 [0.842–0.857],0.285 [0.265–0.304]


,Macro-F1,Se,Sp,PPV
D-1,+0.152 [+0.138; +0.166] P=1.000,-0.063 [-0.093; -0.035] P=0.000,+0.101 [+0.094; +0.108] P=1.000,+0.386 [+0.358; +0.415] P=1.000
D-0,+0.169 [+0.153; +0.183] P=1.000,-0.124 [-0.155; -0.096] P=0.000,+0.115 [+0.108; +0.121] P=1.000,+0.534 [+0.506; +0.562] P=1.000
v1 paket,-0.026 [-0.033; -0.020] P=0.000,+0.017 [+0.000; +0.033] P=0.971,-0.028 [-0.032; -0.023] P=0.000,-0.038 [-0.047; -0.030] P=0.000


,v3.1 paket,D-1,D-0,v1 paket
AF/AFL,77.0% [74–80] (n=725),70.6% [67–74] (n=725),64.6% [61–68] (n=725),78.6% [76–82] (n=725)
sinus takikardia,50.4% [48–53] (n=1233),2.0% [1–3] (n=1233),1.3% [1–2] (n=1233),57.6% [55–60] (n=1233)
sinus bradikardia,5.0% [4–6] (n=1648),5.6% [5–7] (n=1648),0.3% [0–1] (n=1648),5.3% [4–6] (n=1648)
sinus normal,2.0% [1–3] (n=1734),0.5% [0–1] (n=1734),0.4% [0–1] (n=1734),3.6% [3–4] (n=1734)
lainnya,8.8% [8–10] (n=4909),1.7% [1–2] (n=4909),1.0% [1–1] (n=4909),11.6% [11–13] (n=4909)
AFL,67.1% [59–74] (n=170),60.6% [54–68] (n=170),52.9% [45–61] (n=170),68.8% [62–76] (n=170)
AF,80.0% [77–83] (n=555),73.7% [70–77] (n=555),68.1% [64–72] (n=555),81.6% [78–85] (n=555)
AT/SVT,80.8% [69–92] (n=52),17.3% [8–29] (n=52),15.4% [6–27] (n=52),80.8% [69–90] (n=52)


,< 100,100–110,110–120,≥ 120
v3.1 paket,5.6% (n=8140),8.5% (n=563),59.1% (n=367),98.2% (n=454)
v1 paket,7.9% (n=8140),10.3% (n=563),82.6% (n=367),94.5% (n=454)
D-0,0.6% (n=8140),1.1% (n=563),1.4% (n=367),4.8% (n=454)
D-1,1.9% (n=8140),1.4% (n=563),2.7% (n=367),7.5% (n=454)


{'H1_macro_f1': True, 'H2_sensitivitas': False, 'H3_sinus_takikardia': True, 'sekunder_noninferior_se': False}


## 4. Keputusan (RANCANGAN bagian 5)

In [5]:
k = {**G["kriteria"], **N["kriteria"]}
k["semua_lolos"] = all(k[x] for x in ("H1_macro_f1", "H2_sensitivitas", "H3_sinus_takikardia", "H4_nsrdb"))
R["keputusan"] = k
print(json.dumps(k, indent=1))

{
 "H1_macro_f1": true,
 "H2_sensitivitas": false,
 "H3_sinus_takikardia": true,
 "sekunder_noninferior_se": false,
 "H4_nsrdb": true,
 "semua_lolos": false
}


## 5. Kesimpulan

In [6]:
t, b, kel, sub = G["titik"], G["banding"]["D-1"], G["kelompok"], G["subkelompok"]
na = N["alarm"]
print(f"""NSRDB (18 orang sehat, ±{R['data']['NSRDB']['window_lolos']*5/3600:.0f} jam): window teralarm v3.1 {na['v3.1 paket']['alarm']*100:.2f}% vs D-1 {na['D-1']['alarm']*100:.2f}%
  (Δ {N['banding_d1_v31']['delta']*100:+.2f} poin, P(Δ<0) = {N['banding_d1_v31']['p_kurang']:.3f}); episode AF palsu/jam v3.1 {N['episode']['v3.1 paket']['palsu_per_jam']:.2f} vs D-1 {N['episode']['D-1']['palsu_per_jam']:.2f}. H4 {'lolos' if k['H4_nsrdb'] else 'GAGAL'}.
Georgia ({R['data']['Georgia']['window_lolos']} EKG lolos cek): Macro-F1 v3.1 {t['v3.1 paket']['Macro-F1'][0]:.3f} vs D-1 {t['D-1']['Macro-F1'][0]:.3f} (P = {b['Macro-F1'][3]:.3f}); Se {t['v3.1 paket']['Se'][0]:.3f} vs {t['D-1']['Se'][0]:.3f} (P = {b['Se'][3]:.3f}); Sp {t['v3.1 paket']['Sp'][0]:.3f} vs {t['D-1']['Sp'][0]:.3f}.
  H1 {'lolos' if k['H1_macro_f1'] else 'GAGAL'}, H2 {'lolos' if k['H2_sensitivitas'] else 'GAGAL'}; non-inferioritas Se margin −0,02 (sekunder): {'lolos' if k['sekunder_noninferior_se'] else 'gagal'}.
  Sinus takikardia: alarm v3.1 {kel['sinus takikardia']['v3.1 paket']['alarm']*100:.1f}% vs D-1 {kel['sinus takikardia']['D-1']['alarm']*100:.1f}%. H3 {'lolos' if k['H3_sinus_takikardia'] else 'GAGAL'}.
  Sinus bradikardia: v3.1 {kel['sinus bradikardia']['v3.1 paket']['alarm']*100:.1f}% vs D-1 {kel['sinus bradikardia']['D-1']['alarm']*100:.1f}%. AFL: Se v3.1 {sub['AFL']['v3.1 paket']['alarm']:.3f} vs D-1 {sub['AFL']['D-1']['alarm']:.3f}.
Keputusan: {'D-1 tetap kandidat (enam dataset independen)' if k['semua_lolos'] else 'ada kriteria yang gagal; status kandidat ditinjau di README'}.""")

NSRDB (18 orang sehat, ±409 jam): window teralarm v3.1 15.00% vs D-1 5.64%
  (Δ -9.36 poin, P(Δ<0) = 1.000); episode AF palsu/jam v3.1 51.89 vs D-1 20.55. H4 lolos.
Georgia (10249 EKG lolos cek): Macro-F1 v3.1 0.691 vs D-1 0.843 (P = 1.000); Se 0.770 vs 0.706 (P = 0.000); Sp 0.877 vs 0.978.
  H1 lolos, H2 GAGAL; non-inferioritas Se margin −0,02 (sekunder): gagal.
  Sinus takikardia: alarm v3.1 50.4% vs D-1 2.0%. H3 lolos.
  Sinus bradikardia: v3.1 5.0% vs D-1 5.6%. AFL: Se v3.1 0.671 vs D-1 0.606.
Keputusan: ada kriteria yang gagal; status kandidat ditinjau di README.


In [7]:
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
U.simpan_json(R, U.CACHE / "ringkasan.json"); print("tersimpan:", U.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v16.4/ringkasan.json
